In [1]:
import warnings
warnings.filterwarnings('ignore')

# 벡터 저장소 검색기 활용 및 검색 성능 평가

# 라이브러리 설치

## 설치하는 라이브러리의 역할

`faiss-cpu`: 벡터 검색 엔진 라이브러리, 여러 개의 문서 조각 중에서 사용자의 질문과 의미가 가장 유사한 문서를 찾는다. 벡터스토어를 구축한다.  
`rank_bm25`: 키워드 기반 검색 알고리즘 라이브러리, 단순 키워드 일치 여부를 넘어서 단어의 희소성과 빈도를 계산해서 점수화한다.    
`kiwipiepy`: 한국어 형태소 분석기 라이브러리, 한국어 문장을 단어 단위로 쪼개고 조사를 정교하게 분리한다.  
`openpyxl`: 엑셀 파일을 읽고 쓰기 위한 라이브러리, 판다스와 함께 자주 사용된다.

In [2]:
# !pip install faiss-cpu rank_bm25 kiwipiepy openpyxl

# 환경 설정

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## 기본 라이브러리

In [4]:
import os, json, re
from glob import glob
from pprint import pprint
import numpy as np
import pandas as pd

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 벡터저장소(VectorStore)

# Chroma 저장소 생성, 문서 관리, 문서 검색

## 벡터저장소 초기화

허깅 페이스 임베딩 모델을 사용해서 Chroma 벡터저장소 만든다.

In [5]:
# 다국어 처리가 뛰어난 BAAI/bge-m3 모델을 사용해서 허깅 페이스 임베딩 모델을 만든다.
# BAAI/bge-m3 모델은 한국어, 영어 등 여러 언어를 동시에 잘 처리하며, 긴 문장도 효과적으로 벡터화할 수 있어 RAG 시스템 구축시 선호되는 모델이다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

# 비어있는 Chroma 벡터저장소 만든다. 비어있는 벡터저장소를 만들 때는 from_documents() 메소드를 사용하지 않는다.
chroma_db = Chroma(
    # 임베딩 모델을 지정한다. BAAI/bge-m3 모델을 사용해서 Chroma 벡터저장소 만들때 문자를 숫자로 바꾸는 임베딩을 한다.
    embedding_function=embeddings_model,
    collection_name='sample',
    persist_directory='./chroma_db',   
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

get() 메소드는 현재 연결된 Chroma 벡터저장소에 저장된 모든 데이터를 추출하거나, 특정 조건에 맞는 데이터를 조회할 때 사용한다.

In [6]:
chroma_db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

## 벡터저장소 관리

Chroma 벡터저장소에는 Document 객체를 저장해야 하므로 Document를 import 한다.

In [7]:
from langchain_core.documents import Document

Chroma 벡터저장소에 저장할 데이터를 준비한다.

In [8]:
# Chroma 벡터저장소에 저장할 원본 데이터
documents = [
    '인공지능은 컴퓨터 과학의 한 분야입니다.',
    '머신러닝은 인공지능의 하위 분야입니다.',
    '딥러닝은 머신러닝의 한 종류입니다.',
    '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
    '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.',
]

# Document 객체를 생성한다. Document 객체에는 부가정보(metadata)와 본문(page_content)가 포함된다.
doc_objects = []
for index, document in enumerate(documents, start=1):
    # print(index, document)
    doc = Document(
        page_content = document,
        metadata = {'source': f'AI_textbook {index}', 'chapter': f'Chapter {index}'}
    )
    # print(doc)
    doc_objects.append(doc)
print(doc_objects)

# Chroma 벡터저장소에 저장되는 Document 객체의 고유 식별자(ID)를 생성한다.
doc_ids = [f'DOC_{i}' for i in range(1, len(doc_objects) + 1)]
print(doc_ids)

[Document(metadata={'source': 'AI_textbook 1', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'), Document(metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'), Document(metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'), Document(metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'), Document(metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


Chroma 벡터저장소에 데이터를 저장한다.

In [9]:
# Chroma 벡터저장소에 데이터(Document 객체)를 추가한다.
# from_documents() 메소드는 벡터저장소를 만듬과 동시에 데이터가 저장되지만 기존 벡터저장소에 새로운 데이터를 추가하려면 add_documents() 메소드를 사용한다.
added_doc_ids = chroma_db.add_documents(
    documents=doc_objects, # 벡터저장소에 저장할 데이터를 지정한다. 저장할 데이터 타입은 Document 객체가 저장된 리스트 타입이어야 한다.
    ids=doc_ids
)

In [10]:
# add_documents() 메소드는 벡터저장소에 데이터를 추가하고 ids를 리턴한다. len() 함수를 사용해서 추가된 데이터의 개수를 얻어올 수 있다.
print(len(added_doc_ids))

5


In [11]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI_textbook 1', 'chapter': 'Chapter 1'},
  {'chapter': 'Chapter 2', 'source': 'AI_textbook 2'},
  {'source': 'AI_textbook 3', 'chapter': 'Chapter 3'},
  {'chapter': 'Chapter 4', 'source': 'AI_textbook 4'},
  {'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}]}

## 유사도 검사를 이용한 문서 검색

주어진 쿼리와 가장 유사한 문서를 유사도가 높은 순서대로 지정한 개수 만큼 반환한다.

similarity_search() 메소드는 벡터저장소를 검색기로 만들지 않은 상태에서 유사도 검색을 해서 유사도가 높은 순서대로 지정한 개수 만큼 얻어온다.

In [12]:
query = '인공지능과 머신러닝의 관계는?'
# chroma 벡터저장소에서 유사도 검색을 한다.
results = chroma_db.similarity_search(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]


## 문서 수정

chroma 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 내용을 덮어씌워서 수정한다.

In [13]:
# 수정할 새로운 문서 객체를 생성한다.
update_document1 = Document(
    page_content = '인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 11'}
)

update_document2 = Document(
    page_content = '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 22'}
)

update_document3 = Document(
    page_content = '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 33'}
)

In [14]:
# 단일 문서 수정
# update_document() 메소드로 수정할 ID 한 개와 수정할 내용을 지정해서 해당 데이터를 교체한다.
chroma_db.update_document(document_id='DOC_1', document=update_document1)

In [15]:
# 여러 문서 일괄 수정
# update_documents() 메소드로 수정할 ID 여러 개와 수정할 내용을 지정해서 해당 데이터를 교체한다. 리스트로 묶어서 넘겨야 한다.
chroma_db.update_documents(ids=['DOC_2', 'DOC_3'], documents=[update_document2, update_document3])

In [16]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
  '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'chapter': 'Chapter 11', 'source': 'AI_textbook'},
  {'chapter': 'Chapter 22', 'source': 'AI_textbook'},
  {'chapter': 'Chapter 33', 'source': 'AI_textbook'},
  {'source': 'AI_textbook 4', 'chapter': 'Chapter 4'},
  {'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}]}

In [17]:
query = '인공지능과 머신러닝의 관계는?'
results = chroma_db.similarity_search(query, k=2)
print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다. [출처: AI_textbook, Chapter 22]
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]


## 문서 삭제

chroma 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 삭제한다.

In [18]:
# delete() 메소드로 삭제할 ID 한 개를 지정하면 해당 ID의 문서 한 개가 삭제된다.
chroma_db.delete(ids='DOC_1')

In [19]:
# delete() 메소드로 삭제할 ID 두 개 이상을 리스트로 묶어서 지정하면 해당 ID의 문서 여러 개가 삭제된다.
chroma_db.delete(ids=['DOC_2', 'DOC_3'])

In [20]:
chroma_db.get()

{'ids': ['DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI_textbook 4', 'chapter': 'Chapter 4'},
  {'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}]}

In [21]:
# delete_collection() 메소드는 테이블 자체를 제거한다. 따라서, 그 안에 저장된 모든 데이터도 삭제된다.
# delete_collection() 메소드 실행 후 get() 메소드를 실행하면 delete_collection() 메소드에 의해서 내용을 확인할 테이블 자체가 삭제되기 때문에 에러가 발생된다.
chroma_db.delete_collection()

## 문서 검색

In [22]:
chroma_db = Chroma(
    embedding_function=embeddings_model,
    collection_name='sample',
    persist_directory='./chroma_db',   
)

added_doc_ids = chroma_db.add_documents(
    documents=doc_objects,
    ids=doc_ids
)

chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI_textbook 1', 'chapter': 'Chapter 1'},
  {'source': 'AI_textbook 2', 'chapter': 'Chapter 2'},
  {'source': 'AI_textbook 3', 'chapter': 'Chapter 3'},
  {'source': 'AI_textbook 4', 'chapter': 'Chapter 4'},
  {'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}]}

similarity_search() 메소드에 filter 속성을 이용해서 메타데이터가 특정 조건을 만족하는 문서만 얻어올 수 있다.

In [23]:
query = '인공지능과 머신러닝의 관계는?'
results = chroma_db.similarity_search(
    query, # 질문
    k=2, # 유사도가 높은 상위 문서 개수
    # 벡터저장소에 저장된 문서들에서 메타데이터의 'source'가 'AI_textbook 3'인 문서들 중에서만 검색을 실행한다.
    filter={'source': 'AI_textbook 3'}
)
print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]


유사도 점수를 함께 얻어온다.  
유사도 점수는 거리 기준으로 점수가 산정되기 때문에 유사도 점수가 낮을수록 더 유사한 것을 의미한다.

similarity_search_with_score() 메소드는 검색 결과로 문서뿐만 아니라 질문과의 거리(유사도)를 반환한다.

In [24]:
query = '인공지능과 머신러닝의 관계는?'
# similarity_search_with_score() 메소드는 검색 결과로 Document 객체와 질문과의 거리를 튜플로 묶어서 리턴한다.
results = chroma_db.similarity_search_with_score(query, k=2)
# print(results[0])

print(f'쿼리: {query}')
print('가장 유사한 문서:')
# results에는 (문서, 거리) 형태의 튜플이 리스트 형태로 저장되어 있다.
for doc, score in results:
    print(f'- 질문과의 거리: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 질문과의 거리: 0.6592
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 질문과의 거리: 0.8327
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]
----------------------------------------------------------------------------------------------------


관련성 점수를 함께 얻어온다.  
유사도 점수는 거리 기준으로 점수가 산정되기 때문에 유사도 점수가 낮을수록 더 유사한 것을 의미하지만 관련성 점수(정확도)는 높을 수록 더 관련성이 높음을 의미한다.

similarity_search_with_relevance_scores() 메소드는 검색 결과로 문서뿐만 아니라 질문과의 관련성 점수를 반환한다.

In [25]:
query = '인공지능과 머신러닝의 관계는?'
# similarity_search_with_relevance_scores 메소드는 검색 결과로 Document 객체와 질문과의 관련성 점수를 튜플로 묶어서 리턴한다.
# 1에 가까워질수록 관련이 높음(유사함)을 의미하고 0에 가까워질수록 관련이 낮음을 의미한다.
results = chroma_db.similarity_search_with_relevance_scores(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for doc, score in results:
    print(f'- 관련성 점수: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 관련성 점수: 0.5339
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 관련성 점수: 0.4112
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]
----------------------------------------------------------------------------------------------------


# FAISS(Facebook AI Similarity Search) 저장소 생성, 문서 관리, 문서 검색

FAISS는 Facebook AI의 유사도 검색 라이브러리로 효율적인 벡터 저장소 검색 및 클러스터링을 위한 오픈소스 벡터저장소이다.

주요 특징  
&nbsp;&nbsp;&nbsp;▶ 대규모 벡터 데이터셋에서 효율적인 검색이 가능하다.  
&nbsp;&nbsp;&nbsp;▶ RAM을 효율적으로 활용해서 대용량 데이터셋 처리가 가능하다.  
&nbsp;&nbsp;&nbsp;▶ GPU 가속을 지원한다.(faiss-gpu 설치가 필요하다.)  
&nbsp;&nbsp;&nbsp;▶ 다양한 인덱싱 알고리즘을 제공하여 속도와 정확도를 조절 가능하다.  

## FAISS를 사용하기 위해 필요한 라이브러리

Meta(Facebook)에서 개발한 벡터 유사도 검색 라이브러리인 FAISS를 사용하기 위해 import 한다.  

In [26]:
import faiss

복잡한 faiss 명령어를 직접 쓰지 않고 add_documents() 메소드나 similarity_search() 같은 LangChain 표준 메소드를 사용하기 위해 FAISS를 import 한다.

In [27]:
from langchain_community.vectorstores import FAISS

메모리 상에서 문서(Document) 데이터를 Key-Value(ID-문서) 형태로 저장하고 관리하는 가장 간단한 인메모리 문서 저장소를 사용하기 위해 import 한다.  
벡터와 연결된 실제 데이터(문서 내용)을 저장하는 메모리 기반 저장소(창고)이다.

In [28]:
from langchain_community.docstore.in_memory import InMemoryDocstore

## 벡터저장소 초기화

FAISS 인덱스 초기화

FAISS는 데이터를 저장하거나 검색할 때 `차원 수가 일치`해야 한다.  
따라서, 저장할 데이터가 몇 차원인지 미리 알아야 하기때문에 아래 코드를 실행해서 1024 개의 숫자로 이루어진 데이터들을 저장할 준비를 한다.

허깅 페이스의 `BAAI/bge-m3` 임베딩 모델을 사용해서 `embeddings_model.embed_query('hello world')` 명령으로 'hello world'라는 문자열을 임베딩(숫자로 변환)한 결과를 len() 함수를 실행해서 개수를 확인하면 `BAAI/bge-m3` 모델의 차원 수와 같다.  

In [29]:
# IndexFlatL2() 메소드는 L2(유클리드) 거리 방식을 사용하는 벡터들의 고차원 공간상의 위치 및 이웃 검색 알고리즘을 관리하는 인메모리 벡터 인덱스 클래스를 만든다.
faiss_index = faiss.IndexFlatL2(len(embeddings_model.embed_query('hello world')))
print('FAISS 인덱스 초기화 완료')

FAISS 인덱스 초기화 완료


`d` 속성은 'dimension'의 약자로 이 인덱스에 저장될 벡터(숫자 배열)의 길이를 얻어온다.

In [30]:
faiss_index.d

1024

FAISS 수치 인덱스를 기반으로, 텍스트 문서 원본과 벡터저장소를 결합하는 객체를 만든다.

In [31]:
# FAISS 클래스의 생성자로 임베딩 모델, 인덱스 객체, 저장소, ID와 문서를 매핑하는 사전을 넘겨서 FAISS 객체를 만든다.
faiss_db = FAISS(
    # 임베딩 모델을 지정한다. BAAI/bge-m3 모델을 사용해서 FAISS 벡터저장소 만들때 문자를 숫자로 바꾸는 임베딩을 한다.
    embedding_function=embeddings_model,
    # 실제 벡터 검색 연산을 담당하는 FAISS의 인덱스 객체를 지정한다.
    index=faiss_index,
    # 검색 결과로 반환할 텍스트 데이터와 메타데이터를 메모리상에 저장하고 관리하는 저장소를 지정한다.
    docstore=InMemoryDocstore(),
    # FAISS의 인덱스와 docstore에 저장된 문서의 고유 ID를 연결하는 사전으로 사용할 객체를 지정한다.
    # '숫자 벡터 1번이 실제 문장 A이다'라는 연결 내용을 기억할 빈 딕셔너리를 준비하고 데이터를 추가함에 따라 매핑 정보가 쌓인다.
    index_to_docstore_id={},
)

현재 FAISS 벡터저장소(faiss_db)에 실제로 저장된 데이터의 총 개수를 확인한다.  
index는 벡터저장소 내부에 실제 숫자 계산과 저장을 담당하는 FAISS 인덱스 엔진에 접근하는 속성이다.  
ntotal는 FAISS 인덱스에 등록된 전체 데이터의 개수를 기억하는 속성이다.

In [32]:
faiss_db.index.ntotal

0

## FAISS  벡터저장소에 문서 추가하기

FAISS 벡터저장소에 저장할 데이터를 준비한다.

In [33]:
# FAISS 벡터저장소에 저장할 원본 데이터
documents = [
    '인공지능은 컴퓨터 과학의 한 분야입니다.',
    '머신러닝은 인공지능의 하위 분야입니다.',
    '딥러닝은 머신러닝의 한 종류입니다.',
    '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
    '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.',
]

# Document 객체를 생성한다. Document 객체에는 부가정보(metadata)와 본문(page_content)가 포함된다.
doc_objects = []
for index, document in enumerate(documents, start=1):
    doc = Document(
        page_content = document,
        metadata = {'source': f'AI_textbook {index}', 'chapter': f'Chapter {index}'}
    )
    doc_objects.append(doc)
print(doc_objects)

# FAISS 벡터저장소에 저장되는 Document 객체의 고유 식별자(ID)를 생성한다.
doc_ids = [f'DOC_{i}' for i in range(1, len(doc_objects) + 1)]
print(doc_ids)

[Document(metadata={'source': 'AI_textbook 1', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'), Document(metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'), Document(metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'), Document(metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'), Document(metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


FAISS 벡터저장소에 데이터를 저장한다.

In [34]:
# add_documents() 메소드는 FAISS 벡터저장소에 새로운 데이터(Document 객체)를 추가한다.
added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)
print(len(added_doc_ids))
print(added_doc_ids)

5
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


In [35]:
faiss_db.index.ntotal

5

## FAISS 벡터저장소에 문서 삭제하기

FAISS 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 삭제한다.

In [36]:
# delete() 메소드로 삭제할 ID 한 개를 지정하면 해당 ID의 문서 한 개가 삭제된다.
# Chroma 벡터저장소에서 delete() 메소드로 데이터 1건을 삭제할 때 []로 묶지 않아도 되지만 FAISS 벡터저장소 1건을 삭제해도 []로 묶어야 한다.
# Chroma 벡터저장소 존재하지 않는 문서의 ID를 지정해도 에러가 발생하지 않았지만 FAISS 벡터저장소 존재하지 않는 문서의 ID를 지정하면 에러가 발생된다.
faiss_db.delete(ids=['DOC_1'])

True

In [37]:
# delete() 메소드로 삭제할 ID 두 개 이상을 리스트로 묶어서 지정하면 해당 ID의 문서 여러 개가 삭제된다.
faiss_db.delete(ids=['DOC_2', 'DOC_3'])

True

reset() 메소드는 메모리에 올려둔 FAISS 객체 구조는 유지하면서 저장된 모든 벡터와 Docstore 내용을 삭제한다.

In [39]:
faiss_db.index.reset()

In [40]:
faiss_db.index.ntotal

0

## FAISS 벡터저장소에 문서 수정하기

Chroma 벡터저장소 update_document() 메소드나 update_documents() 메소드를 사용해서 직접 문서를 수정할 수 있지만 FAISS 벡터저장소는 직접적으로 문서를 수정하는 메도를 제공하지 않는다.



In [41]:
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)
faiss_db.index.ntotal

5

Chroma 벡터저장소 get() 메소드를 실행하면 벡터저장소에 저장된 모든 데이터를 볼 수 있다. FAISS 벡터저장소는 get() 메소드를 제공하지 않기 때문에 데이터가 저장되는 docstore에서 `_dict` 속성으로 실제 저장된 데이터에 접근해서 values() 메소드를 실행하면 FAISS 벡터저장소에 접근된 모든 데이터를 확인할 수 있다.

In [42]:
list(faiss_db.docstore._dict.values())

[Document(id='DOC_1', metadata={'source': 'AI_textbook 1', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'),
 Document(id='DOC_2', metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'),
 Document(id='DOC_3', metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'),
 Document(id='DOC_4', metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'),
 Document(id='DOC_5', metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]

기존 ID 삭제 후 수정된 문서 재등록 한다.

In [43]:
faiss_db.delete(ids=['DOC_1'])
update_document1 = Document(
    page_content = '인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 11'}
)
added_doc_ids = faiss_db.add_documents(documents=[update_document1], ids=['DOC_1'])

## FAISS 벡터저장소에 문서 검색하기

similarity_search() 메소드로 유사도로 문서 검색

In [44]:
query = '인공지능과 머신러닝의 관계는?'
# FAISS 벡터저장소에서 유사도 검색을 한다.
results = faiss_db.similarity_search(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]


filter를 지정한 문서 검색

In [45]:
query = '인공지능과 머신러닝의 관계는?'
results = faiss_db.similarity_search(query, k=2, filter={'source': 'AI_textbook 3'})

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]


similarity_search_with_score() 메소드로 문서 및 유사도 점수 검색

In [46]:
query = '인공지능과 머신러닝의 관계는?'
results = faiss_db.similarity_search_with_score(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for doc, score in results:
    print(f'- 질문과의 거리: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 질문과의 거리: 0.6592
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 질문과의 거리: 0.6784
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]
----------------------------------------------------------------------------------------------------


similarity_search_with_relevance_scores() 메소드로 문서 및 관련성 점수 검색

In [47]:
query = '인공지능과 머신러닝의 관계는?'
results = faiss_db.similarity_search_with_relevance_scores(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for doc, score in results:
    print(f'- 관련성 점수: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 관련성 점수: 0.5339
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 관련성 점수: 0.5203
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]
----------------------------------------------------------------------------------------------------


## 로컬로 저장 및 로드하기

docstore 속성에 InMemoryDocstore 객체를 이용해서 메모리 상에 임시로 존재하는 벡터저장소를 컴퓨터의 물리적인 공간에 파일로 저장한다.

컴퓨터의 물리적인 공간에 저장하면 아래와 같은 2개의 파일이 생성된다.  
`index.faiss`: 검색을 위한 벡터 데이터가 저장되는 파일  
`index.pkl`: 문서의 본문 내용과 메타데이터가 저장되는 피클 파일

In [48]:
# save_local() 메소드의 인수로 FAISS 벡터저장소의 데이터가 저장될 경로를 넘겨서 저장한다.
faiss_db.save_local('faiss_db')

In [49]:
# FAISS 벡터저장소의 index에서 reset() 메소드를 실행하면 FAISS 벡터저장소의 docstore와 연결된 인덱스가 제거된다.
faiss_db.index.reset()
faiss_db.index.ntotal

0

In [50]:
# FAISS 벡터저장소의 docstore._dict에서 clear()를 실제 저장된 본문과 메타데이터가 제거된다.
faiss_db.docstore._dict.clear()
list(faiss_db.docstore._dict.values())

[]

컴퓨터의 물리적인 공간에 파일로 저장된 FAISS 벡터저장소를 메모리로 불러온다.

In [51]:
# load_local() 메소드의 인수로 FAISS 벡터저장소가 저장된 폴더, 임베딩 모델, 피클 파일 역직렬화 허용 여부를 넘겨서 메모리로 불러온다.
faiss_db = FAISS.load_local(
    # 메모리로 불러올 파일이 저장된 폴더의 (save_local() 메소드에서 지정한)경로를 지정한다.
    folder_path='faiss_db',
    # 메모리로 불러올 FAISS 벡터저장소를 만들때 사용한 인코딩 방식을 지정한다.
    embeddings=embeddings_model,
    # FAISS 벡터저장소 저장 포맷인 피클 파일을 역직렬화하는 것을 허용한다는 의미로 True로 지정한다.
    allow_dangerous_deserialization=True
)

In [52]:
faiss_db.index.ntotal
list(faiss_db.docstore._dict.values())

[Document(id='DOC_2', metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'),
 Document(id='DOC_3', metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'),
 Document(id='DOC_4', metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'),
 Document(id='DOC_5', metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'),
 Document(id='DOC_1', metadata={'source': 'AI_textbook', 'chapter': 'Chapter 11'}, page_content='인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.')]

# RAG 검색기

# Semantic Search(의미 기반 검색) - VectorStore Retriever

의미 기반 검색

텍스트를 벡터 공간에 매핑하여 의미적 유사성을 계산해서 쿼리의 문자 그대로의 의미가 아닌, 의도와 맥락을 이해하여 검색을 수행한다.

작동 원리  
&nbsp;&nbsp;&nbsp;▶ 문서 임베딩: 모든 문서를 문서 조각으로 변환해서 벡터스토어에 사전에 훈련된 언어 모델을 사용하여 임베딩을 생성해서 저장한다.  
&nbsp;&nbsp;&nbsp;▶ 쿼리 임베딩: 사용자의 검색 쿼리도 문서 임베딩과 동일한 방식으로 변환한다.  
&nbsp;&nbsp;&nbsp;▶ 유사도 계산: 코사인 유사도나 유클라디안 거리를 사용해서 쿼리 벡터와 문서 벡터 간의 유사도를 계산한다.  
&nbsp;&nbsp;&nbsp;▶ 결과 반환: 가장 유사한 문서들을 검색 결과로 반환한다.

장점  
&nbsp;&nbsp;&nbsp;▶ 동의어, 관련어 등을 고려한 더 정확한 검색 결과를 제공한다.  
&nbsp;&nbsp;&nbsp;▶ 언어의 느낌과 맥락을 이해하여 검색한다.  
&nbsp;&nbsp;&nbsp;▶ 키워드 기반 검색에서 놓칠 수 있는 정보도 검색 가능하다.

한계  
&nbsp;&nbsp;&nbsp;▶ 대규모 데이터셋에서 계산 비용이 높을 수 있다.  
&nbsp;&nbsp;&nbsp;▶ 임베딩 모델의 품질에 크게 의존한다.  
&nbsp;&nbsp;&nbsp;▶ 매우 특정한 키워드 검색에서는 전통적인 방법보다 성능이 떨어질 수 있다.

## 벡터저장소 초기화

한국어 텍스트 파일들을 불러와서 BAAI/bge-m3 모델의 토크나이저를 기준으로 검색하기 좋게 작게 문서 조각(청크)를 만드는 전처리

불러오려는 텍스트 파일의 목록을 넘겨받아 파일을 읽어서 하나로 합쳐 리턴하는 함수를 선언한다.

In [53]:
def load_txt_files(korean_txt_files):
    data = []
    for txt_file in korean_txt_files:
        loader = TextLoader(txt_file, encoding='utf-8')
        data += loader.load()
    return data

불러오려는 파일 목록을 얻어와서 하나로 합쳐주는 함수를 실행한다.

In [54]:
korean_txt_files = glob(os.path.join('./data', '*_KR.txt'))
print(korean_txt_files)
korean_data = load_txt_files(korean_txt_files)
korean_data

['./data\\리비안_KR.txt', './data\\테슬라_KR.txt']


[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.\n\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.\n'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 

불러온 한글 문서를 문서 조각으로 만들기 위해서 토크나이저와 스플리터를 설정한다.

In [55]:
# from_pretrained() 메소드로 허깅 페이스가 학습시켜 제공하는 BAAI/bge-m3 모델에 적용된 토크나이저를 자동으로 가져와서 토크나이저를 설정한다.
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3')

# from_huggingface_tokenizer() 메소드의 인수로 허깅 페이스 토크나이저, 구분자, 청크 크기, 청크간 겹치는 정도를 넘겨서 스플리터를 설정한다.
text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    # 허깅 페이스의 'BAAI/bge-m3' 모델에 사용한 토크나이저를 토크나이저로 지정한다.
    tokenizer=tokenizer,
    # 텍스트를 나눌 구분자를 정규 표현식을 사용해서 구두점(마침표, 느낌표, 쉼표) 뒤 공백이 한 개이상 나오는 지점으로 지정한다.
    separator=r'(?<=[.!?])\s+',
    # 분할되는 문서 조각(청크)의 최대 크기를 100 토큰으로 제한한다.
    chunk_size=100,
    # 문맥 절단을 방지하기 위해 이전 문서 조각의 끝부분 일부가 다음 문서 조각의 시작 부분에 겹치는 정도를 지정한다.
    chunk_overlap=0,
    # separator에서 설정한 구분자가 단순 문자열인지 정규 표현식인지 알려준다.
    is_separator_regex=True,
    # 텍스트를 나눈 후, 분할의 기준이 되었던 구분자를 버릴지 유지할지 알려준다.
    keep_separator=False
)

스플리터로 읽어온 문서를 분할한다.

In [56]:
# split_documents()의 인수로 텍스트 파일에서 읽어와 하나로 합친 텍스트를 넘겨서 문서 조각으로 분할한다.
korean_docs = text_splitter.split_documents(korean_data)
print(len(korean_docs))
korean_docs

6


[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.'),
 Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.'),
 Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다.2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다.'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_cont

문서 조각으로 나눠진 한국어 문서들을 Chroma 벡터저장소로 저장한다.

In [59]:
# 토크나이저를 만들 때 'BAAI/bge-m3' 모델에서 사용한 토크나이저를 지정했으므로 임베딩 모델도 'BAAI/bge-m3' 모델로 지정한다.
embedding_huggingface = HuggingFaceEmbeddings(model='BAAI/bge-m3')

# from_documents() 메소드로 벡터저장소에 저장할 문서, 임베딩 모델, 테이블 이름, 저장될 경로를 넘겨서 Chroma 벡터저장소를 만든다.
chroma_db = Chroma.from_documents(
    # 읽어들인 한국어 문서가 문서 조각으로 분할된 벡터저장소에 저장할 문서를 지정한다.
    documents=korean_docs,
    # 벡터저장소에 문서 조각을 저장할 때 숫자로 변경할 임베딩 모델을 지정한다.
    embedding=embedding_huggingface,
    collection_name='korean_db',
    persist_directory='./chroma_db',
    # 벡터 간의 유사도 측정에 사용했던 기본값(l2, 유클리드 거리)을 의미적 유사성 판단에 더 유리한 방식인 코사인 유사도 방식을 사용한다고 설정한다.
    # 코사인 유사도는 RAG 텍스트 검색에서 가장 흔하게 사용된다.
    collection_metadata={'hnsw:space': 'cosine'}
)

len(chroma_db.get()['ids']) # Chroma 벡터저장소에 저장된 문서 조각의 개수

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

6

In [58]:
# chroma_db.delete_collection()

## 벡터검색기 초기화

검색된 문서가 실제 쿼리와 얼마나 비슷한지 유사도를 계산하기 위해서 cosine_similarity를 import 한다.

In [60]:
from langchain_community.utils.math import cosine_similarity

Chroma 벡터저장소를 벡터검색기로 만들어서 사용자 질문에 가장 적합한(유사도가 높은) 문서 조각을 얻어온다.

In [61]:
# as_retriever() 메소드에 질문에 가장 적합한 문서를 가져올 개수(기본값은 4)를 넘겨서 벡터저장소를 벡터검색기로 만든다.
chroma_k_retirever = chroma_db.as_retriever(search_kwargs={'k': 2})

query = '리비안은 언제 사업을 시작했나요?'
# invoke() 메소드의 인수로 질문을 넘겨서 질문에 가장 적합한 문서들을 가져온다.
retirever_docs = chroma_k_retirever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    # cosine_similarity() 함수의 인수로 질문과 답변을 임베딩된 결과를 리스트 형태로 넘겨서 코사인 유사도를 계산한다.
    score = cosine_similarity(
        [embeddings_model.embed_query(query)], # 질문
        [embeddings_model.embed_query(doc.page_content)] # 답변
    )[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5992
----------------------------------------------------------------------------------------------------


## 벡터검색기에 유사도 점수의 임계값(Threshold) 지정

`search_type` 속성은 벡터 저장소에서 문서를 `어떤 전략`으로 찾아올 것인가를 지정한다.  
`similarity`가 기본값으로 사용되며 가장 표준적인 검색 방식으로 사용자 질문과 가장 유사한 문서 조각을 유사도 점수 상위 k개를 가져온다. 엉뚱한 답을 할 수 있다.  
`similarity_score_threshold`는 설정한 임계값을 넘는 문서 조각을 유사도 점수 상위 k개를 가져온다. 임계값을 너무 높게잡으면 결과가 빈번하게 누락된다.  
`mmr`은 다양성을 고려한 검색 방식으로 질문과 유사한 문서 조각을 찾되, 이미 찾은 문서들과 내용이 너무 중복되는 문서는 제외한다.

In [62]:
chroma_threshold_retirever = chroma_db.as_retriever(
    # 설정한 임계값을 넘는 문서 조각을 유사도 점수 상위 k개를 가져온다.
    search_type='similarity_score_threshold',
    search_kwargs={
        'k': 2,
        # 코사인 유사도 점수가 0.6을 넘는 결과만 반환하도록 설정한다. 코사인 유사도는 높을수록 좋다. 낮은 유사도의 무의미한 결과는 버린다.
        # 임계값 설정을 사용하려면 search_type='similarity_score_threshold' 속성을 지정해야 한다.
        'score_threshold': 0.6
    }
)

query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = chroma_threshold_retirever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------


## 벡터검색기 MMR(Maximal Marginal Relevance) 검색

MMR은 RAG 시스템에서 검색 결과의 `관련성(Relevance)`과 `다양성(Diversity)`을 동시에 고려하여 최적의 문서 집합을 선별하는 `중복 제거` 알고리즘이다.  
단순히 쿼리와 가장 유사한 상위 k개 문서만 가져오면, 거의 비슷한 내용의 문서들이 중복으로 뽑히는 정보의 `중복성(Redundancy)` 문제가 발생합니다. MMR은 이러한 한계를 극복하기 위해 등장했다.

search_kwargs 속성에 최종적으로 사용자에게 보여줄(반환할) 문서의 개수(`k`), MMR 알고리즘을 적용하기 위해 벡터저장소에서 뽑아낼 후보군의 개수(`fetch_k`), 다양성과 관련성의 비율(`lambda_mult`)을 지정한다. 

In [63]:
chroma_mmr = chroma_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        # 최종적으로 사용자에게 보여줄(반환할) 문서의 개수를 지정한다.
        'k': 3,
        # MMR 알고리즘을 적용하기 위해 벡터저장소에서 뽑아낼 후보군 개수를 지정한다.
        # 'fetch_k'에 지정한 6개 중에서 중복되지 않는 최적의 'k'에 지정한 3개를 고르게 된다. 'fetch_k' 값은 'k' 보다 크거나 같아야 한다.
        'fetch_k': 6,
        # 다양성과 관련성의 비율을 지정한다.
        # 1.0에 가까울수록 질문과의 '유사도'만 따져서 기본 검색과 비슷해지고 0.0에 가까울수록 이미 뽑힌 결과와 얼마나 다른가 즉, '다양성'을 더 중요하게 여긴다.
        'lambda_mult': 0.5 # 기본값은 0.5
    }
)

query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = chroma_mmr.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5992
----------------------------------------------------------------------------------------------------
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다.회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다.테슬라는 2010년 6월 나스닥에 상장되었습니다.2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다.
{'source': './data\\테슬라_KR.txt'}
코사인 유사도: 0.2862
------------------